In [1]:
from pathlib import Path
import os

PROJECT_ROOT_CACHE = Path.cwd().resolve()
if PROJECT_ROOT_CACHE.name.lower() == 'notebooks':
    PROJECT_ROOT_CACHE = PROJECT_ROOT_CACHE.parent
HF_HOME_DIR = PROJECT_ROOT_CACHE / 'models' / 'huggingface'
HF_HUB_DIR = HF_HOME_DIR / 'hub'
HF_HUB_DIR.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(HF_HOME_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_HUB_DIR)
os.environ['SENTENCE_TRANSFORMERS_HOME'] = str(HF_HUB_DIR)

print('Hugging Face 模型将下载到:', HF_HUB_DIR)
print('必须在导入 transformers / sentence_transformers 之前运行本 Cell。')


Hugging Face 模型将下载到: D:\dev\projects\fourlang_translation\models\huggingface\hub
必须在导入 transformers / sentence_transformers 之前运行本 Cell。


In [2]:
from pathlib import Path
import importlib
import importlib.metadata as metadata
import sys

PROJECT_ROOT_BOOT = Path.cwd().resolve()
if PROJECT_ROOT_BOOT.name.lower() == 'notebooks':
    PROJECT_ROOT_BOOT = PROJECT_ROOT_BOOT.parent
HF_COMPAT_DIR = PROJECT_ROOT_BOOT / '.hf_compat'
if not HF_COMPAT_DIR.exists():
    raise FileNotFoundError(f'兼容依赖目录不存在: {HF_COMPAT_DIR}')

compat_path = str(HF_COMPAT_DIR)
if compat_path in sys.path:
    sys.path.remove(compat_path)
sys.path.insert(0, compat_path)
importlib.invalidate_caches()

versions = {name: metadata.version(name) for name in ('tokenizers', 'transformers', 'sentence-transformers')}
print('使用隔离 Hugging Face 依赖:', HF_COMPAT_DIR)
print('兼容版本:', versions)
assert versions == {'tokenizers': '0.20.3', 'transformers': '4.46.3', 'sentence-transformers': '3.3.1'}
print('依赖路径与版本正常。')


使用隔离 Hugging Face 依赖: D:\dev\projects\fourlang_translation\.hf_compat
兼容版本: {'tokenizers': '0.20.3', 'transformers': '4.46.3', 'sentence-transformers': '3.3.1'}
依赖路径与版本正常。


In [3]:
import importlib.metadata as metadata
import subprocess
import sys

PINNED_PACKAGES = {
    'tokenizers': '0.20.3',
    'transformers': '4.46.3',
    'sentence-transformers': '3.3.1',
}

def package_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

installed = {name: package_version(name) for name in PINNED_PACKAGES}
print('当前包版本:', installed)
needs_repair = any(installed[name] != version for name, version in PINNED_PACKAGES.items())

if needs_repair:
    specs = [f'{name}=={version}' for name, version in PINNED_PACKAGES.items()]
    print('正在修复 Hugging Face 包元数据与兼容版本:', specs)
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--ignore-installed', '--no-deps', '--no-cache-dir', *specs
    ])
    print('修复完成。现在请重启 Notebook Kernel，再从 Cell 1 重新运行全部 Cell。')
else:
    print('依赖版本正常，可以继续运行。')


当前包版本: {'tokenizers': '0.20.3', 'transformers': '4.46.3', 'sentence-transformers': '3.3.1'}
依赖版本正常，可以继续运行。


In [4]:
from pathlib import Path
import csv
import getpass
import hashlib
import io
import json
import os
import random
import re
import time
import zipfile
from itertools import zip_longest

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'en_uz' / 'opus_public_v1'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'clean' / 'en_uz' / 'public_5k_v1'
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_ACCEPTED = 5000
PRESELECT_FOR_LLM = 6500
MAX_CHARS = 320
MAX_FILTERED_PER_SOURCE = {
    'Tatoeba': 2000,
    'tldr-pages': 4000,
    'wikimedia': 60000,
}
LABSE_MODEL = 'sentence-transformers/LaBSE'
LABSE_BATCH_SIZE = 32
LABSE_MIN_SCORE = 0.62
FINAL_LABSE_MIN_SCORE = 0.70
FINAL_LLM_MIN_CONFIDENCE = 0.88

SOURCES = [
    {
        'source': 'Tatoeba',
        'version': 'v2026-07-08',
        'url': 'https://object.pouta.csc.fi/OPUS-Tatoeba/v2026-07-08/moses/en-uz.txt.zip',
        'license': 'CC BY 2.0 FR',
        'commercial_status': 'attribution_required',
        'homepage': 'https://opus.nlpl.eu/datasets/Tatoeba',
    },
    {
        'source': 'tldr-pages',
        'version': 'v2026-07-07',
        'url': 'https://object.pouta.csc.fi/OPUS-tldr-pages/v2026-07-07/moses/en-uz.txt.zip',
        'license': 'CC BY 4.0 (upstream pages; verify OPUS package metadata)',
        'commercial_status': 'attribution_and_metadata_review_required',
        'homepage': 'https://github.com/tldr-pages/tldr',
    },
    {
        'source': 'wikimedia',
        'version': 'v20260327',
        'url': 'https://object.pouta.csc.fi/OPUS-wikimedia/v20260327/moses/en-uz.txt.zip',
        'license': 'Upstream Wikimedia terms vary; commonly CC BY-SA/GFDL',
        'commercial_status': 'attribution_sharealike_legal_review_required',
        'homepage': 'https://opus.nlpl.eu/datasets/wikimedia',
    },
]

SOURCE_MANIFEST_PATH = OUTPUT_DIR / 'source_manifest.json'
SOURCE_MANIFEST_PATH.write_text(
    json.dumps({'created_for': 'en-uz public-data candidate extraction', 'sources': SOURCES}, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('目标:', TARGET_ACCEPTED, '条；LLM 只审核候选:', PRESELECT_FOR_LLM, '条')
print('注意：source_manifest.json 是来源记录，不是法律意见。商业发布前仍需确认归属与许可证履行方式。')


PROJECT_ROOT: D:\dev\projects\fourlang_translation
OUTPUT_DIR: D:\dev\projects\fourlang_translation\data\clean\en_uz\public_5k_v1
目标: 5000 条；LLM 只审核候选: 6500 条
注意：source_manifest.json 是来源记录，不是法律意见。商业发布前仍需确认归属与许可证履行方式。


D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
SPACE_RE = re.compile(r'\s+')
URL_OR_HTML_RE = re.compile(r'(https?://|www\.|<[^>]+>|&(?:nbsp|amp|lt|gt);)', re.I)
CYRILLIC_RE = re.compile(r'[\u0400-\u04FF]')
LATIN_RE = re.compile(r'[A-Za-z]')
NUMBER_RE = re.compile(r'(?<!\w)[+-]?\d+(?:[.,]\d+)*(?!\w)')
WORD_RE = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿʻʼ’'`-]+")
REPEATED_RE = re.compile(r'\b(\w{2,})(?:\s+\1){2,}\b', re.I)

def normalize_text(text):
    text = str(text).replace('\ufeff', '').replace('\u00a0', ' ')
    return SPACE_RE.sub(' ', text).strip()

def normalized_key(text):
    text = normalize_text(text).lower()
    return re.sub(r'[^\w]+', '', text, flags=re.UNICODE)

def number_signature(text):
    return sorted(x.replace(',', '.') for x in NUMBER_RE.findall(text))

def cheap_pair_filter(en, uz):
    en, uz = normalize_text(en), normalize_text(uz)
    if not en or not uz:
        return False, 'empty'
    if not (3 <= len(en) <= MAX_CHARS and 3 <= len(uz) <= MAX_CHARS):
        return False, 'length'
    if URL_OR_HTML_RE.search(en) or URL_OR_HTML_RE.search(uz):
        return False, 'url_or_html'
    if normalized_key(en) == normalized_key(uz):
        return False, 'identical'
    if CYRILLIC_RE.search(uz):
        return False, 'uz_not_latin_script'
    en_letters = LATIN_RE.findall(en)
    if len(en_letters) < 2:
        return False, 'en_not_latin'
    en_words, uz_words = WORD_RE.findall(en), WORD_RE.findall(uz)
    if not (2 <= len(en_words) <= 55 and 1 <= len(uz_words) <= 65):
        return False, 'word_count'
    ratio = max(len(en), len(uz)) / max(1, min(len(en), len(uz)))
    if ratio > 3.6:
        return False, 'length_ratio'
    if number_signature(en) != number_signature(uz):
        return False, 'number_mismatch'
    if REPEATED_RE.search(en.lower()) or REPEATED_RE.search(uz.lower()):
        return False, 'repetition'
    return True, 'ok'

def download_cached(source):
    filename = source['url'].rsplit('/', 1)[-1]
    target = RAW_DIR / f"{source['source']}_{source['version']}_{filename}"
    if target.exists() and target.stat().st_size > 1000:
        print('使用缓存:', target.name)
        return target
    partial = target.with_suffix(target.suffix + '.part')
    print('下载:', source['url'])
    with requests.get(source['url'], stream=True, timeout=(20, 180)) as response:
        response.raise_for_status()
        total = int(response.headers.get('content-length', 0))
        with partial.open('wb') as f, tqdm(total=total, unit='B', unit_scale=True, desc=source['source']) as bar:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))
    partial.replace(target)
    return target

def choose_language_member(names, lang):
    ignored = ('readme', 'license', '.xml', '.ces', '.ids')
    candidates = [
        n for n in names
        if n.lower().endswith('.' + lang) and not any(x in n.lower() for x in ignored)
    ]
    if not candidates:
        raise FileNotFoundError(f'ZIP 内找不到 .{lang} 文本；成员示例: {names[:20]}')
    preferred = [n for n in candidates if 'en-uz' in n.lower() or 'uz-en' in n.lower()]
    return sorted(preferred or candidates, key=lambda x: (len(x), x))[0]

def stream_filtered_reservoir(zip_path, source, limit):
    rng = random.Random(SEED + sum(map(ord, source['source'])))
    kept, passed, raw_count = [], 0, 0
    rejected = {}
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        en_name = choose_language_member(names, 'en')
        uz_name = choose_language_member(names, 'uz')
        print(source['source'], '成员:', en_name, '|', uz_name)
        with io.TextIOWrapper(zf.open(en_name), encoding='utf-8', errors='replace') as en_f, \
             io.TextIOWrapper(zf.open(uz_name), encoding='utf-8', errors='replace') as uz_f:
            iterator = zip_longest(en_f, uz_f, fillvalue=None)
            for line_no, pair in enumerate(tqdm(iterator, desc=f"过滤 {source['source']}"), start=1):
                en, uz = pair
                if en is None or uz is None:
                    rejected['unaligned_file_length'] = rejected.get('unaligned_file_length', 0) + 1
                    break
                raw_count += 1
                en, uz = normalize_text(en), normalize_text(uz)
                ok, reason = cheap_pair_filter(en, uz)
                if not ok:
                    rejected[reason] = rejected.get(reason, 0) + 1
                    continue
                passed += 1
                row = {
                    'pair_id': hashlib.sha1(f"{source['source']}|{source['version']}|{line_no}|{en}|{uz}".encode('utf-8')).hexdigest()[:20],
                    'en': en,
                    'uz': uz,
                    'source': source['source'],
                    'source_version': source['version'],
                    'source_line': line_no,
                    'license': source['license'],
                    'commercial_status': source['commercial_status'],
                    'source_url': source['url'],
                }
                if len(kept) < limit:
                    kept.append(row)
                else:
                    j = rng.randrange(passed)
                    if j < limit:
                        kept[j] = row
    stats = {'source': source['source'], 'raw': raw_count, 'passed_cheap_filter': passed, 'reservoir_kept': len(kept), **{f'reject_{k}': v for k, v in rejected.items()}}
    return kept, stats


In [6]:
RUN_PUBLIC_EXTRACTION = True
FILTERED_POOL_PATH = OUTPUT_DIR / 'public_filtered_pool.csv'
FILTER_STATS_PATH = OUTPUT_DIR / 'filter_stats.csv'

if RUN_PUBLIC_EXTRACTION:
    all_rows, all_stats = [], []
    for source in SOURCES:
        archive = download_cached(source)
        rows, stats = stream_filtered_reservoir(archive, source, MAX_FILTERED_PER_SOURCE[source['source']])
        all_rows.extend(rows)
        all_stats.append(stats)

    pool_df = pd.DataFrame(all_rows)
    if pool_df.empty:
        raise RuntimeError('公开语料过滤后为空，请查看下载文件和过滤统计。')
    pool_df['dedupe_key'] = pool_df['en'].map(normalized_key) + '|' + pool_df['uz'].map(normalized_key)
    before = len(pool_df)
    pool_df = pool_df.drop_duplicates('dedupe_key', keep='first').drop(columns='dedupe_key').reset_index(drop=True)
    pool_df.to_csv(FILTERED_POOL_PATH, index=False, encoding='utf-8-sig')
    pd.DataFrame(all_stats).to_csv(FILTER_STATS_PATH, index=False, encoding='utf-8-sig')
    print('去重:', before, '->', len(pool_df))
else:
    if not FILTERED_POOL_PATH.exists():
        raise FileNotFoundError(f'请先把 RUN_PUBLIC_EXTRACTION=True 运行本 Cell：{FILTERED_POOL_PATH}')
    pool_df = pd.read_csv(FILTERED_POOL_PATH, keep_default_na=False)

display(pd.read_csv(FILTER_STATS_PATH, keep_default_na=False).fillna(0))
print(pool_df.groupby('source').size().sort_values(ascending=False))
display(pool_df.sample(min(10, len(pool_df)), random_state=SEED)[['en', 'uz', 'source']])


使用缓存: Tatoeba_v2026-07-08_en-uz.txt.zip
Tatoeba 成员: Tatoeba.en-uz.en | Tatoeba.en-uz.uz


过滤 Tatoeba: 514it [00:00, 72335.00it/s]


使用缓存: tldr-pages_v2026-07-07_en-uz.txt.zip
tldr-pages 成员: tldr-pages.en-uz.en | tldr-pages.en-uz.uz


过滤 tldr-pages: 1378it [00:00, 57603.96it/s]


使用缓存: wikimedia_v20260327_en-uz.txt.zip
wikimedia 成员: wikimedia.en-uz.en | wikimedia.en-uz.uz


过滤 wikimedia: 1372671it [01:06, 20733.82it/s]


去重: 61703 -> 56563


,source,raw,passed_cheap_filter,reservoir_kept,reject_uz_not_latin_script,reject_word_count,reject_number_mismatch,reject_url_or_html,reject_length_ratio,reject_length,reject_identical,reject_en_not_latin,reject_repetition
0,Tatoeba,514,351,351,158.0,2.0,3.0,,,,,,
1,tldr-pages,1378,1352,1352,,,,22.0,4.0,,,,
2,wikimedia,1372671,667482,60000,5791.0,110040.0,393075.0,15210.0,23267.0,140898.0,13980.0,2846.0,82.0


source
wikimedia     55882
Tatoeba         350
tldr-pages      331
dtype: int64


,en,uz,source
38351,Kumdere (Kurdish: Şibêbiyê) is a neighbourhood...,Kumdere (Kurdish) Turkiyaning Mardin viloyati ...,wikimedia
9462,"In 1856, the Republic of Maryland requested mi...",1856-yilda Merilend Respublikasi Merilend ko'c...,wikimedia
26794,Pauline Parmentier (French pronunciation: ​[pɔ...,Pauline Parmentier ( Fransuzcha talaffuzi: pɔl...,wikimedia
54731,The metre accounts for about 3% of surviving a...,Hisoblagich qadimgi va klassik arab oyatlarini...,wikimedia
40476,"Reuters. 14 March 2023. ↑ ""Iran to participate...",Reuters (14-mart 2023-yil). ↑ „Iran to partici...,wikimedia
54121,"As of 8 May 2022, Hazan is the official nation...","2022-yil, 8-may holatiga ko‘ra, Hazan Isroil t...",wikimedia
22864,Paris Saint-Germain F.C. 31 July 2022.,Paris Saint-Germain F.C. (31-iyul 2022-yil).,wikimedia
41393,Mount Nashiwari is within the town's boundaries.,Nashivari tog'i shahar chegarasida joylashgan.,wikimedia
43773,"On June 14, 2003, she and Acuña met again, thi...",2003-yil 14-iyunda u va Akuna yana Buenos-Ayre...,wikimedia
25565,"Village in Erzincan Province, Turkey Kurutilek...",Kurutilek Qishloq Kurutilek Village Country Tu...,wikimedia


In [7]:
RUN_LABSE_SCORING = True
SCORED_PATH = OUTPUT_DIR / 'public_candidates_scored.csv'
LLM_QUEUE_PATH = OUTPUT_DIR / 'public_llm_review_queue.csv'

if RUN_LABSE_SCORING:
    import torch
    from sentence_transformers import SentenceTransformer

    if 'pool_df' not in globals():
        pool_df = pd.read_csv(FILTERED_POOL_PATH, keep_default_na=False)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('LaBSE 设备:', device, '；这是本地推理排序，不是训练。')
    print('第一次会下载模型；8GB 显存建议保持 batch_size=', LABSE_BATCH_SIZE)
    labse = SentenceTransformer(LABSE_MODEL, device=device)
    en_emb = labse.encode(
        pool_df['en'].tolist(), batch_size=LABSE_BATCH_SIZE, normalize_embeddings=True,
        show_progress_bar=True, convert_to_numpy=True,
    )
    uz_emb = labse.encode(
        pool_df['uz'].tolist(), batch_size=LABSE_BATCH_SIZE, normalize_embeddings=True,
        show_progress_bar=True, convert_to_numpy=True,
    )
    pool_df = pool_df.copy()
    pool_df['labse_score'] = np.einsum('ij,ij->i', en_emb, uz_emb).astype('float32')
    pool_df = pool_df.sort_values(['labse_score', 'pair_id'], ascending=[False, True]).reset_index(drop=True)
    pool_df.to_csv(SCORED_PATH, index=False, encoding='utf-8-sig')

    eligible = pool_df[pool_df['labse_score'] >= LABSE_MIN_SCORE].copy()
    if len(eligible) < PRESELECT_FOR_LLM:
        print(f'警告：LaBSE >= {LABSE_MIN_SCORE} 只有 {len(eligible)} 条，将全部送审。')
    queue_df = eligible.head(PRESELECT_FOR_LLM).copy()
    queue_df.to_csv(LLM_QUEUE_PATH, index=False, encoding='utf-8-sig')

    del labse, en_emb, uz_emb
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    if not SCORED_PATH.exists() or not LLM_QUEUE_PATH.exists():
        raise FileNotFoundError('请先将 RUN_LABSE_SCORING=True 运行本 Cell。')
    pool_df = pd.read_csv(SCORED_PATH, keep_default_na=False)
    queue_df = pd.read_csv(LLM_QUEUE_PATH, keep_default_na=False)

print('进入 LLM 审核队列:', len(queue_df))
print(queue_df.groupby('source').size().sort_values(ascending=False))
display(queue_df['labse_score'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))
display(queue_df.head(10)[['en', 'uz', 'source', 'labse_score']])


LaBSE 设备: cuda ；这是本地推理排序，不是训练。
第一次会下载模型；8GB 显存建议保持 batch_size= 32


D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\dev\projects\fourlang_translation\models\huggingface\hub\models--sentence-transformers--LaBSE. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not instal

进入 LLM 审核队列: 6500
source
wikimedia     6452
Tatoeba         46
tldr-pages       2
dtype: int64


count    6500.000000
mean        0.965121
std         0.009868
min         0.952701
1%          0.952878
5%          0.953609
25%         0.957374
50%         0.962860
75%         0.970528
95%         0.985246
99%         0.995665
max         0.998868
Name: labse_score, dtype: float64

,en,uz,source,labse_score
0,"She traps Spider-Man in the vault, reveals her...","She traps Spider-Man in the vault, reveals her...",wikimedia,0.998868
1,Shannon Daddy Day Camp Kim Hinton Protect and ...,Shannon Daddy Day Camp Kim Hinton Protect and ...,wikimedia,0.998722
2,1 2 (in Dutch) Jan & Andries Both biography in...,1 2 (niderlandcha) Jan & Andries Both biograph...,wikimedia,0.998649
3,She apologizes for her previous actions and re...,She apologizes for her previous actions and re...,wikimedia,0.998442
4,↑ Kawanishi Town official statistics (in Japan...,↑ Kawanishi Town official statistics (yaponcha...,wikimedia,0.997902
5,"She has appeared in multiple photo books, movi...","She has appeared in multiple photo books, movi...",wikimedia,0.997661
6,↑ Kiyosu City official statistics (in Japanese...,↑ Kiyosu City official statistics (yaponcha) ↑...,wikimedia,0.997626
7,Firozpur Zira Guru Har Sahai,Firozpur Zira Guru Har Sahay,wikimedia,0.997372
8,Diehl Defence GmbH & Co.,Diehl Defense GmbH & Co.,wikimedia,0.997333
9,Karan Kapoor Kapoor in August 2016 Born (1962-...,Karan Kapoor Kapoor in August 2016 Born (1962-...,wikimedia,0.997242


In [8]:
from openai import OpenAI

DASHSCOPE_BASE_URL = 'https://dashscope.aliyuncs.com/compatible-mode/v1'
AUDIT_MODEL = 'qwen-max'
AUDIT_BATCH_SIZE = 25
AUDIT_PATH = OUTPUT_DIR / 'qwen_audit_results.csv'
AUDIT_USAGE_PATH = OUTPUT_DIR / 'qwen_audit_usage.json'

def extract_json_object(text):
    text = (text or '').strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.I)
        text = re.sub(r'\s*```$', '', text)
    start, end = text.find('{'), text.rfind('}')
    if start < 0 or end <= start:
        raise ValueError('模型响应中没有完整 JSON 对象')
    return json.loads(text[start:end + 1])

def validate_audit_payload(payload, expected_ids):
    items = payload.get('results')
    if not isinstance(items, list):
        raise ValueError('JSON 必须含 results 数组')
    parsed = {}
    for item in items:
        pair_id = str(item.get('pair_id', ''))
        label = str(item.get('label', '')).lower()
        if pair_id not in expected_ids or label not in {'correct', 'minor', 'wrong', 'junk'}:
            continue
        try:
            confidence = min(1.0, max(0.0, float(item.get('confidence', 0))))
        except Exception:
            confidence = 0.0
        parsed[pair_id] = {
            'pair_id': pair_id,
            'llm_label': label,
            'llm_confidence': confidence,
            'llm_reason': normalize_text(str(item.get('reason', '')))[:120],
        }
    missing = set(expected_ids) - set(parsed)
    if missing:
        raise ValueError(f'模型漏审 {len(missing)} 条: {sorted(missing)[:3]}')
    return [parsed[x] for x in expected_ids]

def audit_batch(client, batch_df, retries=4):
    records = batch_df[['pair_id', 'en', 'uz']].to_dict('records')
    prompt = f"""你是英语-乌兹别克语平行语料质检员。逐条判断乌兹别克语是否自然、是否为现代乌兹别克语（拉丁字母），以及与英语是否完整等义。
标签只能是：
correct = 自然且完整等义；minor = 基本等义但有轻微语法、措辞或遗漏；wrong = 含义明显不一致、严重遗漏或并非乌兹别克语；junk = 噪声、乱码、列表碎片、模板残片。
数字、否定、专名和单位不一致不能标 correct。不要改写或生成翻译。
只输出 JSON：{{"results":[{{"pair_id":"...","label":"correct|minor|wrong|junk","confidence":0.0,"reason":"错误时给极短原因，correct 时留空"}}]}}
必须返回全部 {len(records)} 个 pair_id，顺序与输入一致。
输入：{json.dumps(records, ensure_ascii=False)}"""
    last_error = None
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=AUDIT_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=0,
                response_format={'type': 'json_object'},
                max_tokens=1800,
            )
            payload = extract_json_object(response.choices[0].message.content)
            rows = validate_audit_payload(payload, [x['pair_id'] for x in records])
            usage = getattr(response, 'usage', None)
            usage_dict = {
                'prompt_tokens': int(getattr(usage, 'prompt_tokens', 0) or 0),
                'completion_tokens': int(getattr(usage, 'completion_tokens', 0) or 0),
                'total_tokens': int(getattr(usage, 'total_tokens', 0) or 0),
            }
            return rows, usage_dict
        except Exception as exc:
            last_error = exc
            if attempt + 1 < retries:
                time.sleep(2 ** attempt)
    raise RuntimeError(f'批量审核连续失败: {last_error}')

print('Qwen 只负责分类审核，不负责创造句子。')
print('预计请求数约:', int(np.ceil(len(queue_df) / AUDIT_BATCH_SIZE)))
print('下一 Cell 默认不会调用 API；确认后把 RUN_LLM_AUDIT 改为 True。')


Qwen 只负责分类审核，不负责创造句子。
预计请求数约: 260
下一 Cell 默认不会调用 API；确认后把 RUN_LLM_AUDIT 改为 True。


In [11]:
RUN_LLM_AUDIT = True

def atomic_csv_save(df, path):
    temp = path.with_suffix(path.suffix + '.tmp')
    df.to_csv(temp, index=False, encoding='utf-8-sig')
    temp.replace(path)

if RUN_LLM_AUDIT:
    api_key = os.getenv('DASHSCOPE_API_KEY') or getpass.getpass('请输入 DashScope API Key（不会写入 Notebook）：')
    if not api_key.strip():
        raise ValueError('API Key 为空')
    client = OpenAI(api_key=api_key.strip(), base_url=DASHSCOPE_BASE_URL)

    queue_df = pd.read_csv(LLM_QUEUE_PATH, keep_default_na=False)
    if AUDIT_PATH.exists():
        audit_df = pd.read_csv(AUDIT_PATH, keep_default_na=False)
    else:
        audit_df = pd.DataFrame(columns=['pair_id', 'llm_label', 'llm_confidence', 'llm_reason'])
    audit_df = audit_df.drop_duplicates('pair_id', keep='last')
    done = set(audit_df['pair_id'].astype(str))
    pending = queue_df[~queue_df['pair_id'].astype(str).isin(done)].reset_index(drop=True)
    print('已完成:', len(done), '待审核:', len(pending), '可安全断点续跑')

    usage_total = {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'completed_batches': 0}
    for start in tqdm(range(0, len(pending), AUDIT_BATCH_SIZE), desc='Qwen 审核'):
        batch = pending.iloc[start:start + AUDIT_BATCH_SIZE]
        rows, usage = audit_batch(client, batch)
        audit_df = pd.concat([audit_df, pd.DataFrame(rows)], ignore_index=True)
        audit_df = audit_df.drop_duplicates('pair_id', keep='last')
        atomic_csv_save(audit_df, AUDIT_PATH)
        for key in ('prompt_tokens', 'completion_tokens', 'total_tokens'):
            usage_total[key] += usage[key]
        usage_total['completed_batches'] += 1
        AUDIT_USAGE_PATH.write_text(json.dumps(usage_total, ensure_ascii=False, indent=2), encoding='utf-8')

if AUDIT_PATH.exists():
    audit_df = pd.read_csv(AUDIT_PATH, keep_default_na=False)
    print('当前已审核:', audit_df['pair_id'].nunique(), '/', len(queue_df))
    display(audit_df['llm_label'].value_counts(dropna=False))
    if AUDIT_USAGE_PATH.exists():
        print('本次运行 Token 用量:', json.loads(AUDIT_USAGE_PATH.read_text(encoding='utf-8')))
else:
    print('尚未调用 API。先检查上一 Cell 的候选样本，然后将 RUN_LLM_AUDIT=True。')


已完成: 0 待审核: 6500 可安全断点续跑


Qwen 审核: 100%|██████████| 260/260 [1:59:17<00:00, 27.53s/it]  

当前已审核: 6500 / 6500


llm_label
correct    5211
minor       788
wrong       330
junk        171
Name: count, dtype: int64

本次运行 Token 用量: {'prompt_tokens': 779287, 'completion_tokens': 284292, 'total_tokens': 1063579, 'completed_batches': 260}


In [12]:
FINAL_PAIRS_PATH = OUTPUT_DIR / 'accepted_public_en_uz_5000.csv'
TRAIN_PAIRS_PATH = OUTPUT_DIR / 'train_pairs.csv'
VALID_PAIRS_PATH = OUTPUT_DIR / 'validation_pairs.csv'
TRAIN_DIRECTED_PATH = OUTPUT_DIR / 'train_directed.jsonl'
VALID_DIRECTED_PATH = OUTPUT_DIR / 'validation_directed.jsonl'
HUMAN_REVIEW_PATH = OUTPUT_DIR / 'human_review_sample.csv'
DATASET_CARD_PATH = OUTPUT_DIR / 'dataset_manifest.json'

def write_directed_jsonl(df, path):
    with path.open('w', encoding='utf-8') as f:
        for row in df.itertuples(index=False):
            common = {
                'pair_id': row.pair_id, 'source': row.source, 'source_version': row.source_version,
                'license': row.license, 'commercial_status': row.commercial_status,
                'labse_score': round(float(row.labse_score), 6),
            }
            for src_lang, tgt_lang, src_text, tgt_text in [
                ('en', 'uz', row.en, row.uz), ('uz', 'en', row.uz, row.en),
            ]:
                record = {**common, 'src_lang': src_lang, 'tgt_lang': tgt_lang, 'src_text': src_text, 'tgt_text': tgt_text}
                f.write(json.dumps(record, ensure_ascii=False) + '\n')

if not AUDIT_PATH.exists():
    print('还没有审核结果，先运行 API 审核 Cell；本 Cell 暂不导出训练集。')
else:
    queue_df = pd.read_csv(LLM_QUEUE_PATH, keep_default_na=False)
    audit_df = pd.read_csv(AUDIT_PATH, keep_default_na=False).drop_duplicates('pair_id', keep='last')
    merged = queue_df.merge(audit_df, on='pair_id', how='left', validate='one_to_one')
    completed = merged[merged['llm_label'].astype(str).ne('') & merged['llm_label'].notna()].copy()
    accepted = completed[
        (completed['llm_label'] == 'correct') &
        (pd.to_numeric(completed['llm_confidence'], errors='coerce') >= FINAL_LLM_MIN_CONFIDENCE) &
        (pd.to_numeric(completed['labse_score'], errors='coerce') >= FINAL_LABSE_MIN_SCORE)
    ].copy()
    accepted['ranking_score'] = (
        pd.to_numeric(accepted['labse_score'], errors='coerce') * 0.55 +
        pd.to_numeric(accepted['llm_confidence'], errors='coerce') * 0.45
    )
    accepted = accepted.sort_values(['ranking_score', 'pair_id'], ascending=[False, True]).head(TARGET_ACCEPTED).reset_index(drop=True)

    print('审核完成:', len(completed), '/', len(queue_df))
    print('满足严格门槛:', len(accepted), '/', TARGET_ACCEPTED)
    if len(completed) < len(queue_df):
        print('审核尚未跑完，可继续断点续跑；当前导出只是临时结果。')
    if len(accepted) < TARGET_ACCEPTED:
        print('未达到 5000：不要自动放宽门槛。可增加公开候选来源/池大小，或人工复核 minor。')

    accepted.to_csv(FINAL_PAIRS_PATH, index=False, encoding='utf-8-sig')
    rng = np.random.default_rng(SEED)
    order = rng.permutation(len(accepted))
    valid_size = min(max(1, round(len(accepted) * 0.05)), len(accepted)) if len(accepted) else 0
    valid_idx = set(order[:valid_size].tolist())
    valid_df = accepted.iloc[sorted(valid_idx)].copy() if valid_idx else accepted.iloc[0:0].copy()
    train_df = accepted.drop(index=sorted(valid_idx)).copy() if valid_idx else accepted.copy()
    train_df.to_csv(TRAIN_PAIRS_PATH, index=False, encoding='utf-8-sig')
    valid_df.to_csv(VALID_PAIRS_PATH, index=False, encoding='utf-8-sig')
    write_directed_jsonl(train_df, TRAIN_DIRECTED_PATH)
    write_directed_jsonl(valid_df, VALID_DIRECTED_PATH)

    errors = completed[completed['llm_label'].isin(['minor', 'wrong', 'junk'])].head(100)
    good_sample = accepted.sample(min(200, len(accepted)), random_state=SEED) if len(accepted) else accepted
    human_review = pd.concat([good_sample, errors], ignore_index=True).drop_duplicates('pair_id')
    human_review.to_csv(HUMAN_REVIEW_PATH, index=False, encoding='utf-8-sig')

    manifest = {
        'dataset_name': 'public_en_uz_5k_v1',
        'target_pairs': TARGET_ACCEPTED,
        'actual_pairs': len(accepted),
        'selection': {
            'cheap_filters': True, 'semantic_model': LABSE_MODEL, 'labse_min': FINAL_LABSE_MIN_SCORE,
            'llm_auditor': AUDIT_MODEL, 'llm_label': 'correct', 'llm_min_confidence': FINAL_LLM_MIN_CONFIDENCE,
            'human_review_required_before_release': True,
        },
        'splits': {'train_pairs': len(train_df), 'validation_pairs': len(valid_df)},
        'sources': SOURCES,
        'warning': 'Public availability and automated quality acceptance do not constitute commercial legal clearance. Preserve attribution/provenance and obtain license review before release.',
    }
    DATASET_CARD_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    print('输出目录:', OUTPUT_DIR)
    print('训练方向记录:', len(train_df) * 2, '验证方向记录:', len(valid_df) * 2)
    display(accepted.groupby(['source', 'license']).size().rename('accepted').reset_index())
    display(accepted.head(10)[['en', 'uz', 'source', 'labse_score', 'llm_confidence']])


审核完成: 6500 / 6500
满足严格门槛: 5000 / 5000
输出目录: D:\dev\projects\fourlang_translation\data\clean\en_uz\public_5k_v1
训练方向记录: 9500 验证方向记录: 500


,source,license,accepted
0,Tatoeba,CC BY 2.0 FR,42
1,tldr-pages,CC BY 4.0 (upstream pages; verify OPUS package...,2
2,wikimedia,Upstream Wikimedia terms vary; commonly CC BY-...,4956


,en,uz,source,labse_score,llm_confidence
0,"↑ Akhavan-Majid, Roya (December 1, 2004).","↑ Akhavan-Majid, Roya (1 December 2004).",wikimedia,0.997110,1.0
1,"""On the Impossibility of Informationally Effic...","""On the Impossibility of Informationally Effic...",wikimedia,0.995960,1.0
2,"V 1908, p. 72","V 1908, s. 72",wikimedia,0.995546,1.0
3,Pamela Landy 2008 Death Race Prison Warden Cla...,Pamela Landy 2008 Death Race Prison Warden Cla...,wikimedia,0.995525,1.0
4,"1986 Los Angeles, U.S. Hard John McEnroe Stefa...","1986 Los Angeles, U.S. Hard John McEnroe Stefa...",wikimedia,0.995303,1.0
5,"Superior People Who Drive Russia Forward"" Apri...","Superior People Who Drive Russia Forward"" Apre...",wikimedia,0.995300,1.0
6,"1974, p. 655 1 2 Poitevin 1927, pp. 2–4 ↑ ""Que...","1974, s. 655 1 2 Poitevin 1927, pp. 2–4 ↑ „Que...",wikimedia,0.995232,1.0
7,"1 2 3 IMAI, Ichiro (March 1982), ""Small Stock ...","1 2 3 IMAI, Ichiro (March 1982), ""Small Stock ...",wikimedia,0.995178,1.0
8,"Translation of a paper entitled ""Die schizoide...","Translation of a paper entitled ""Die schizoide...",wikimedia,0.995105,1.0
9,"1983 Los Angeles, U.S. Hard John McEnroe Sandy...","1983 Los Angeles, U.S. Hard John McEnroe Sandy...",wikimedia,0.993789,1.0
